- **학습 주제**
  - **환경 설정 및 API 호출**: 환경 변수 관리와 OpenAI API를 활용한 LLM 호출 방법 이해
  - **프롬프팅 기법과 합성 데이터 생성**: 다양한 프롬프팅 기법(Zero-shot, Few-shot, CoT)을 활용한 고품질 합성 데이터 생성
  - **LLM as Judge를 통한 데이터 평가**: LLM 기반 자동 평가 시스템 설계 및 

- **Step 요약**
  - **Step 1** : 환경 설정 및 API 기본 호출 - 환경 변수로 API 키를 관리하고, OpenAI 호환 API를 통해 LLM을 호출하는 기본 구조 이해
  - **Step 2** : 프롬프팅 기법 비교 - Zero-shot, Few-shot, CoT 프롬프팅의 특성을 비교하고 각 기법의 장단점 이해
  - **Step 3** : 구조화된 합성 데이터 생성 - 프롬프팅 기법을 활용하여 JSON 형식의 구조화된 합성 데이터 생성
  - **Step 4** : 합성 데이터 평가 (LLM as Judge) - LLM을 평가자로 활용하여 생성된 합성 데이터의 품질을 자동으로 평가하는 시스템 구현

In [ ]:
### TODO 1: 환경 변수 설정 및 API 키 로드

# 1. load_dotenv() 함수를 사용하여 .env 파일을 로드하세요
load_dotenv(".env") #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 2. os.getenv() 함수를 사용하여 GMS_KEY를 가져오세요
GMS_KEY = os.getenv("GMS_KEY") #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 정상적으로 잘 작동하는지 확인해봅시다.
if GMS_KEY:
    print("Success API Key Setting!")
else:
    print("Failed to load API Key. Please check your .env file.")

In [ ]:
# LLM 호출 함수 구현
# 1. OpenAI 클라이언트 생성
client = OpenAI(
    api_key=GMS_KEY,
    base_url="https://gms.ssafy.io/gmsapi/api.openai.com/v1/"
)

# 2. chat_completion 함수를 완성
def chat_completion(prompt: str,
                    system_prompt: str = None,
                    model: str = "gpt-5-nano") -> str:
    """
    LLM을 호출하여 응답을 반환하는 함수

    Args:
        prompt: 사용자 메시지
        system_prompt: 시스템 메시지 (선택)
        model: 사용할 모델명

    Returns:
        LLM의 응답 텍스트
    """
    messages = []

    # 시스템 프롬프트는 LLM의 동작을 제어하는 프롬프트로
    # 대화의 스타일, 성격, 제약 조건 등을 결정합니다.
    # LLM이 어떤 방식으로 답변해야 하는지를 미리 정해주는 지침으로 이해하면 됩니다.
    # 필수는 아니기 때문에 `if`문으로 처리하였습니다.
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# 잘 작동하는지 테스트해볼까요?
test_response = chat_completion("안녕하세요! 간단한 인사말을 해주세요.")
print("테스트 응답:")
print(test_response)

In [ ]:
#TODO2: 프롬프팅 기법 비교 실험
# 공통 시스템 프롬프트
SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다.
추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다."""

user_query = "스릴러 영화를 추천해줘"

# 1. Zero-shot 프롬프트 `zero_shot_prompt`를 작성하세요 (예시 없이 직접 지시)
zero_shot_prompt = f"{user_query}" #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

print("=" * 50)
print("Zero-shot 프롬프팅 결과:")
print("=" * 50)
zero_shot_result = chat_completion(zero_shot_prompt, SYSTEM_PROMPT)
print(zero_shot_result)
print()

# 2. Few-shot 프롬프트 `few_shot_prompt`를 작성하세요 (2-3개의 예시 포함)
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
few_shot_prompt = f'''다음은 영화 추천 예시입니다:

질문: 로맨스 영화를 추천해줘
답변: 영화 제목: 노트북 (The Notebook)
개봉 연도: 2004년
추천 이유: 시대를 초월한 순수한 사랑 이야기로, 감동적인 스토리와 아름다운 영상미가 돋보이는 클래식 로맨스 영화입니다.

질문: 코미디 영화를 추천해줘
답변: 영화 제목: 행오버 (The Hangover)
개봉 연도: 2009년
추천 이유: 총각파티 후 기억을 잃은 친구들의 황당한 모험을 그린 코미디로, 예측 불가능한 전개와 유쾌한 유머가 가득합니다.

질문: {user_query}
답변:'''
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

print("=" * 50)
print("Few-shot 프롬프팅 결과:")
print("=" * 50)
few_shot_result = chat_completion(few_shot_prompt, SYSTEM_PROMPT)
print(few_shot_result)
print()

# 3. CoT 프롬프트 `cot_prompt`를 작성하세요 (단계별 추론 유도)
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
cot_prompt = f'''{user_query}
영화를 추천하기 전에 다음 단계를 따라 생각해주세요:

1단계: 스릴러 장르의 핵심 요소가 무엇인지 정의합니다 (긴장감, 반전, 서스펜스 등)
2단계: 이 요소들을 잘 갖춘 대표적인 스릴러 영화들을 떠올립니다
3단계: 그 중에서 가장 추천할 만한 영화 1개를 선택하고 이유를 설명합니다

위 단계를 따라 추론 과정을 보여주고, 최종 추천을 해주세요.'''
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
print("=" * 50)
print("Chain-of-Thought 프롬프팅 결과:")
print("=" * 50)
cot_result = chat_completion(cot_prompt, SYSTEM_PROMPT)
print(cot_result)
print()

In [ ]:
#JSON 파싱 유틸리티 함수

def json_parsing(output_text: str) -> dict:
    """
    LLM 응답에서 JSON 부분을 추출하여 딕셔너리로 변환

    Args:
        output_text: LLM의 응답 텍스트

    Returns:
        파싱된 딕셔너리
    """
    try:
        # ```json ... ``` 형식에서 JSON 추출
        if "```json" in output_text:
            output_text = output_text[output_text.index("```json") + len("```json"):].strip()
            output_text = output_text[:output_text.index("```")].strip()
        return json.loads(output_text)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"JSON 파싱 오류: {e}")
        return None

In [ ]:
#TODO 3: 구조화된 합성 데이터 생성
# 1. 구조화된 출력을 위한 시스템 프롬프트를 작성하세요
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
STRUCTURED_GENERATOR_SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

## 1. 입력 형식
[추천받고자 하는 영화 장르]

## 2. 작업 지시
- 요청된 장르에 가장 적합한 영화 1개를 추천합니다.
- 추천 이유는 구체적이고 설득력 있게 작성합니다.
- 친근하고 유머러스한 말투로 설명합니다.

## 3. 출력 형식
- 출력 형식은 다음 포맷을 따릅니다.
```json
{
    "movie_name": [영화 이름],
    "year": [개봉 연도],
    "genre": [장르],
    "reason": [추천 이유]
}
```
"""
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


# 2. 1에서 만든 프롬프트를 활용하여 답변을 받고,
# 답변을 JSON으로 파싱하여 `dict` 형태로 반환하도록 함수를 구성해주세요.
def generate_movie_recommendation(genre: str, temperature: float = 1.0) -> dict:
    """
    특정 장르에 대한 영화 추천 데이터를 생성

    Args:
        genre: 영화 장르
        temperature: 다양성 조절 파라미터 (0.0~2.0)

    Returns:
        구조화된 영화 추천 데이터
    """
    #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": STRUCTURED_GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": f"{genre} 영화를 추천해줘"}            
        ],
        temperature=temperature
    )

    output = response.choices[0].message.content
    return json_parsing(output)
    #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 3. 3가지 다른 장르에 대해 합성 데이터를 생성하고 `synthetic_data`에 저장해주세요
# #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★.
genres = ["공포", "SF", "액션"]
synthetic_data = []

for genre in genres:
    print(f"\n{genre} 장르 데이터 생성 중...")
    data = generate_movie_recommendation(genre, temperature=1.0)
    if data:
        synthetic_data.append(data)
        print("생성 완료: ", end=""); pprint(data)
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 생성된 데이터 확인
print("\n" + "=" * 50)
print("생성된 합성 데이터:")
print("=" * 50)
for i, data in enumerate(synthetic_data, 1):
    print(f"\n[{i}] ", end=""); pprint(data)

In [ ]:
# LLM as Judge
# 평가자 시스템 프롬프트입니다.
# 데이터를 생성할 때 사용한 입력 프롬프트와 생성된 데이터, 그리고 평가기준을 토대로
# 어떤 방식으로 평가할지 지정합니다.

# 작성 요령은 함수 정의서를 쓰는 것처럼 변수명을 지정하여 작성합니다.
# 이렇게 작성하면 추후에 다른 평가 기준이나 새로운 모델 답변에 대해 유연하게 프롬프트를 활용할 수 있습니다.
# 평가 또한 JSON 형태로 출력하도록 지정합니다.
JUDGE_SYSTEM_PROMPT = """당신의 역할은 모델 답변 자동 평가자입니다. 입력 프롬프트와 모델 답변을 보고, 평가 기준에 따라 모델 답변을 평가합니다.

## 1. 입력 형식
    - 입력 프롬프트: [instruction]
    - 모델 답변: [output]
    - 평가 기준: [criteria]

## 2. 작업 지시
    - [instruction]에 따른 모델 결과물인 [output]을 평가합니다.
    - [output]이 [criteria]를 충족하는지 평가합니다.

## 3. 채점 원칙 (각 기준별 1–5점, 정수만)
    - 5점 (탁월): 기준을 완전히 충족. 오류·누락 없음. 구체적이고 실행가능.
    - 4점 (우수): 대체로 충족. 사소한 흠만 있음(정확성·구체성·형식 등에서 경미한 누락).
    - 3점 (보통): 핵심은 맞지만 눈에 띄는 약점 존재(누락, 모호함, 근거 부족 등).
    - 2점 (미흡): 중요한 요구를 여러 곳에서 놓침 또는 오류/비논리 다수.
    - 1점 (부적합): 전반적으로 요청과 어긋남, 의미있는 도움/근거 없음, 안전·정책 위반 가능성.

## 4. 출력 형식 (엄격 준수)
    - "score"는 1–5점의 정수로 평가한다.
    - "comment"는 한국어 1–3문장으로 평가한다. 구체적이고 실행 가능하게 작성한다.
    - 출력 형식은 JSON 형식인 <output_format>을 준수한다.

<output_format>
```json
{
    "score": [모델의 답변 평가 점수],
    "comment": [평가 주석]
}
```
</output_format>
"""

# 평가자 프롬프트입니다.
# 주어진 입력 프롬프트, 생성된 데이터, 평가 기준을 제공해줍니다.
# `.format`문을 통해 변수를 채워줄 예정이기 때문에 아래와 같이 작성됩니다.
JUDGE_USER_PROMPT_TEMPLATE = """- 입력 프롬프트: {instruction}
- 모델 답변: {output}
- 평가 기준: {criteria}
"""


def evaluate_with_llm(instruction: str, output: str, criteria: str) -> dict:
    """
    LLM as Judge를 사용하여 모델 출력을 평가

    Args:
        instruction: 원본 지시문
        output: 평가할 모델 출력
        criteria: 평가 기준

    Returns:
        평가 결과 (score, comment)
    """
    # 평가자 프롬프트로 실제로 해당 데이터를 만들 때 사용한 프롬프트, 해당 데이터와 평가기준을 넣어
    # 완성된 평가자 프롬프트를 생성합니다.
    user_prompt = JUDGE_USER_PROMPT_TEMPLATE.format(
        instruction=instruction,
        output=output,
        criteria=criteria
    )

    # 위에서 생성한 평가자 프롬프트와 평가자 시스템프롬프트를 기입하여 응답을 생성합니다.
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
    )

    result = response.choices[0].message.content
    return json_parsing(result)


# 앞서 생성한 합성 데이터 중 하나를 평가해봅시다
test_data = synthetic_data[0]
print("평가 대상 데이터:")
pprint(test_data)

# 데이터를 생성할 때 만든 프롬프트 (`instruction`)
# 위의 프롬프트를 통해 생성한 데이터 (`output`)
# 해당 출력을 평가할 `criteria`
instruction = STRUCTURED_GENERATOR_SYSTEM_PROMPT
output = json.dumps(test_data, ensure_ascii=False)
criteria = "요청의 충실도: 요청된 장르에 맞는 영화를 추천했는지, 필수 정보(제목, 연도, 이유)가 모두 포함되었는지 평가"
print("\n평가 중...\n")

evaluation_result = evaluate_with_llm(instruction, output, criteria)
print("평가 결과:")
pprint(evaluation_result)

In [ ]:
#TODO 4: 평가 결과 분석 및 해석
# 1. 모든 합성 데이터에 대해 평가를 수행하여 `evaluation_results`에 저장해주세요.
criteria_list = [
    "요청의 충실도: 요청된 장르에 맞는 영화를 추천했는지, 필수 정보가 모두 포함되었는지",
    "추천 이유의 구체성: 추천 이유가 구체적이고 설득력 있는지",
    "정보의 정확성: 영화 제목과 개봉 연도가 실제와 일치하는지"
]
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
evaluation_results = []
print("모든 합성 데이터 평가 시작...\n")

for i, data in enumerate(synthetic_data):
    print(f"[{i+1}] {data.get('movie_name', 'Unknown')} 평가 중...")

    data_output = json.dumps(data, ensure_ascii=False)

    # 각 기준에 대해 평가
    scores = []
    for criteria in criteria_list:
        result = evaluate_with_llm(
            instruction=STRUCTURED_GENERATOR_SYSTEM_PROMPT,
            output=data_output,
            criteria=criteria
        )
        if result and "score" in result:
            scores.append(result["score"])

    avg_score = sum(scores) / len(scores) if scores else 0
    evaluation_results.append({
        "data": data,
        "scores": scores,
        "average_score": avg_score
    })
    print(f"   평균 점수: {avg_score:.2f}")
    #★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# 2. 평가 결과를 정리하여 출력하세요
print("\n" + "=" * 60)
print("평가 결과 요약")
print("=" * 60)

#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
for i, result in enumerate(evaluation_results, 1):
    data = result["data"]
    print(f"\n[{i}] {data.get('movie_name', 'Unknown')} ({data.get('genre', 'Unknown')})")
    print(f"    개별 점수: {result['scores']}")
    print(f"    평균 점수: {result['average_score']:.2f}")
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


# 2.1 평균 점수와 품질 분석 결과를 출력하세요
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
total_avg = sum(r["average_score"] for r in evaluation_results) / len(evaluation_results) if evaluation_results else 0

print("\n" + "=" * 60)
print("품질 분석 결과")
print("=" * 60)
print(f"전체 평균 점수: {total_avg:.2f} / 5.00")

if total_avg >= 4.0:
    print("품질 등급: 우수 - 합성 데이터가 높은 품질을 보입니다.")
elif total_avg >= 3.0:
    print("품질 등급: 보통 - 일부 개선이 필요합니다.")
else:
    print("품질 등급: 미흡 - 프롬프트 또는 생성 파라미터 조정이 필요합니다.")
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


# 2.2 결과를 `pandas.DataFrame`으로 저장해주세요.
#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
df = []
for r in evaluation_results:
    df.append({
        "movie_name": r["data"]["movie_name"],
        "year": r["data"]["year"],
        "genre": r["data"]["genre"],
        "reasons": r["data"]["reason"],
        "Score 1": r["scores"][0],
        "Score 2": r["scores"][1],
        "Score 3": r["scores"][2],
        "Average Score": sum(r["scores"]) / 3,
    })
pd.DataFrame(df)

#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★